# Leyenda - Livrable 3 - Captioning

## Architectures schématiques

Pour nos architectures GRU et LSTM, nous avons utilisé une architecture de type CNN pour le pré-traitement des images. Nous avons décidé d'utiliser InceptionV3 (avec ImageNet) pour en tant que CNN afin de transformer les images en des représentations numériques informatives (embeddings visuels). Ces représentations sont ensuite utilisées comme entrées pour les modèles de langage (GRU et LSTM) qui génèrent des légendes pour les images.

L'avantage d'utiliser InceptionV3 (avec ImageNet) est de pouvoir identifier à la fois des détails fins mais aussi des motifs globaux. Ils seront ensuite traduits en vecteurs compréhensibles par le modèle de langage. En utilisant un CNN pré-entraîné, nous pouvons tirer parti de la puissance de l'apprentissage profond sans avoir besoin d'un grand ensemble de données d'images pour entraîner notre propre CNN à partir de zéro. Cela nous permet également de bénéficier des connaissances acquises par le modèle sur un large éventail d'images et de classes.

L'objectif principal de cette étape est de compresser les informations visuelles tout en mettant en relief les éléments clés des images. Cela permet de ne pas surcharger la mémoire lorsqu'il y'a des milliers d'images à traiter.

En amont de cela, nous avons effectué le pré-traitement des images, qui consiste à redimensionner les images à une taille fixe (299x299 pixels) et à les normaliser pour que les valeurs des pixels soient comprises entre 0 et 1. Cela permet de garantir que toutes les images ont la même taille et la même échelle de valeurs, ce qui est essentiel pour l'entraînement du modèle.

Pour le pré-traitement du texte, nous avons effectué les étapes suivantes :

- Nettoyage du texte :

        Passage en minuscules, suppression de la ponctuation si nécessaire, nettoyage des caractères spéciaux.

- Ajout de tokens spéciaux :

        Chaque phrase est entourée de deux tokens : <start> (début) et <end> (fin), pour aider le modèle à comprendre quand commencer et arrêter la génération.

- Tokenisation :

        On convertit chaque mot en entier unique, grâce à un dictionnaire (tokenizer) construit sur l’ensemble des légendes du dataset.

- Padding :

        Les séquences (phrases) sont de tailles variables, donc on les complete (padding) avec des zéros pour qu’elles aient toutes la même longueur.

- Embedding :

        Ces entiers sont ensuite convertis en vecteurs denses à l’aide d’une Embedding layer, qui apprend à capturer la signification sémantique des mots.

### Architecture avec GRU

<img src="./../figures/Architecture_captionning_GRU.png" alt="Architecture_captionning_GRU" style="width: 800px;"/>

### Architecture avec LSTM

<img src="./../figures/Architecture_captionning_LSTM.png" alt="Architecture_captionning_LSTM" style="width: 800px;"/>

## Code des modèles de captionning

Nous allons entraîner deux modèles de réseaux de neurones récurrents, LSTM et GRU, sur notre jeu de données de captioning d’images. L’objectif est de comparer leurs performances en termes de qualité des légendes générées, de rapidité d’entraînement et de capacité à généraliser. Cette comparaison nous permettra de déterminer lequel des deux modèles est le plus adapté à notre tâche.

Le modèle GRU (Gated Recurrent Unit) est une version simplifiée du LSTM. Il fusionne certaines portes pour réduire le nombre de paramètres, ce qui le rend plus rapide et moins coûteux en ressources. Il est souvent aussi performant que le LSTM, surtout sur des jeux de données de taille moyenne ou lorsque les séquences ne sont pas trop longues. 

Le modèle LSTM (Long Short-Term Memory) est conçu pour capturer les dépendances à long terme dans les séquences. Il utilise une cellule de mémoire interne et trois portes (oubli, entrée, sortie) pour gérer l'information de manière fine. Il est particulièrement adapté aux séquences longues et complexes, mais peut être plus lent à entraîner en raison de sa structure plus lourde.



### Modèle avec GRU

### Modèle avec LSTM

## Analyse des résultats

### Courbes

### Tableau des performances

### Conclusion

## Exemples de légendes

## Pistes d'amélioration

Plusieurs pistes peuvent être envisagées pour améliorer la qualité des légendes générées.


L’utilisation du Beam Search permettrait d’améliorer la stratégie de prédiction. En génération de texte, la méthode la plus simple (dite greedy) consiste à choisir à chaque étape le mot ayant la plus forte probabilité. Cependant, cette approche peut mener à des séquences sous-optimales. Le Beam Search propose d’explorer plusieurs séquences candidates en parallèle (déterminées par un paramètre appelé beam width) et de conserver uniquement les plus prometteuses à chaque étape. Cette stratégie permet de générer des phrases plus cohérentes et souvent plus proches du sens attendu.

L’entraînement du modèle pourrait être optimisé en s’appuyant sur un plus grand volume de données, ou en appliquant de l’augmentation de données à plusieurs reprises, afin d’accroître la diversité visuelle et améliorer la capacité de généralisation du modèle. Aujourd'hui, nous avons testé les modèles avec 6000 images au maximum mais cela n'est pas le plus performant. Dans l'avenir utiliser les 80000 images du dataset MS COCO serait plus intéressant afin d'optimiser notre modèle.

# Leyenda - Livrable 3 - Pipeline

## Initialisation

In [ ]:
from src.model_loader import ModelLoader
from src.data_loader import DataLoader
from src.utils import get_class_weigths
from src.utils import show_clean_and_noisy_images
from src.utils import clean_invalid_images
from src.utils import filter_by_custom_binary_model

import os
import tensorflow as tf
import sys

project_root = os.path.abspath(os.path.join(".."))  # One level up from current script
if project_root not in sys.path:
    sys.path.append(project_root)
    
    
data_loader = DataLoader()

classification_model_loader = ModelLoader(model_name="classification_Inception")

autoencoder_skiplayer_loader = ModelLoader(model_name="autoencoder_skiplayer")

## Chargement et traitement des données

In [ ]:
directory_to_load = "../datasets/Test"

In [ ]:
clean_invalid_images(directory_to_load)

## Pipeline de données

### 1ère étape : Classification photo/pas photo
Chargement de notre modèle le plus performant : Inception sur le dataset binaire avec class weight

In [ ]:
inception_model = classification_model_loader.create_model_with_inception(show_summary=False, init_weigths_path="./../models/weights/inception/binary_cw-20250411-145213.weights.h5")

Activation de la fonction pour séparer les photos en utilisant le modèle de classification à partir du dossier Test

In [ ]:
filter_by_custom_binary_model(inception_model, directory_to_load, "./../datasets/Photo_filtered", threshold=0.7, max_images=100)

### 2ème étape : Traitement des photos
Création du modèle autoencodeur avec ses poids

In [ ]:
denoising_autoencoder = autoencoder_skiplayer_loader.create_skip_layer_autoencoder(show_summary=False, init_weigths_path="../models/weights/autoencoder/autoencoder_skip_layers_mae_tanguy.weights.h5")

Utilisation du modèle pour traiter les images

In [ ]:
photo_denoised_path = "./../datasets/Photo_denoised"
from src.utils import denoise_images

denoise_images(
    input_dir="./../datasets/Photo_filtered",
    output_dir=photo_denoised_path,
    model=denoising_autoencoder,
    target_size=(256, 256)  # à adapter si ton modèle utilise une autre taille
)